# Plan d.viii -- Robustness Checks

Re-estimates the **first-differenced OLS** -- the primary model for H1/H2 inference, redesignated
2026-07-17 (`outputs/modeling_path_decision.csv` addendum; `research_plan.md` §a Update note) --
under three alternate conditions, one at a time, and reports how much the DIVP/DIVM coefficients
move. This is the evidentiary basis for how much confidence to place in H1/H2: a coefficient that
survives all three checks in sign, magnitude, and significance supports confidence; one that
flips or loses significance under a specific check must be flagged and discussed, not smoothed
over.

See `docs/2_plan/analysis/viii_robustness_checks.md` for the full spec (including its own
2026-07-17 Update note making the same redesignation explicit before this notebook was written).

**Every "primary model" reference below means the first-differenced OLS**, per that
redesignation -- not Model B/ARDL, which the plan doc originally anticipated before step vi/vii's
evidence came in.

**Input** (`outputs/`):
- `analysis_frame_1990_2024.csv` (35 obs, from step i) -- baseline and check 2/3 sample
- `analysis_frame_1990_2023.csv` (34 obs, from step i) -- check 1 sample
- `model_b_first_differenced_ols_coefficients.csv` / `..._fit_stats.csv`,
  `model_diff_hac_coefficients.csv` (from step vi/vii) -- baseline sanity-check targets

**Outputs** (`outputs/`):
- `robustness_check1_1990_2023_coefficients.csv` / `..._fit_stats.csv` / `..._hac_coefficients.csv`
- `robustness_check2_excl_shock_coefficients.csv` / `..._fit_stats.csv` / `..._hac_coefficients.csv`
- `robustness_check3_raw_fdi_coefficients.csv` / `..._fit_stats.csv` / `..._hac_coefficients.csv`
- `robustness_checks_comparison.csv` -- the single required comparison table: baseline + checks
  1-3 + check 4 (marked N/A), DIVP/DIVM coefficients and significance (original and HAC) side by
  side, per model

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tools import add_constant

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

FRAME_2024_IN = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
FRAME_2023_IN = OUTPUT_DIR / "analysis_frame_1990_2023.csv"

BASELINE_COEF_IN = OUTPUT_DIR / "model_b_first_differenced_ols_coefficients.csv"
BASELINE_FIT_IN = OUTPUT_DIR / "model_b_first_differenced_ols_fit_stats.csv"
BASELINE_HAC_IN = OUTPUT_DIR / "model_diff_hac_coefficients.csv"

CHECK1_COEF_OUT = OUTPUT_DIR / "robustness_check1_1990_2023_coefficients.csv"
CHECK1_FIT_OUT = OUTPUT_DIR / "robustness_check1_1990_2023_fit_stats.csv"
CHECK1_HAC_OUT = OUTPUT_DIR / "robustness_check1_1990_2023_hac_coefficients.csv"

CHECK2_COEF_OUT = OUTPUT_DIR / "robustness_check2_excl_shock_coefficients.csv"
CHECK2_FIT_OUT = OUTPUT_DIR / "robustness_check2_excl_shock_fit_stats.csv"
CHECK2_HAC_OUT = OUTPUT_DIR / "robustness_check2_excl_shock_hac_coefficients.csv"

CHECK3_COEF_OUT = OUTPUT_DIR / "robustness_check3_raw_fdi_coefficients.csv"
CHECK3_FIT_OUT = OUTPUT_DIR / "robustness_check3_raw_fdi_fit_stats.csv"
CHECK3_HAC_OUT = OUTPUT_DIR / "robustness_check3_raw_fdi_hac_coefficients.csv"

COMPARISON_OUT = OUTPUT_DIR / "robustness_checks_comparison.csv"

SIGNIFICANCE_LEVELS = [(0.01, "***"), (0.05, "**"), (0.10, "*")]


def stars(p_value: float) -> str:
    for threshold, mark in SIGNIFICANCE_LEVELS:
        if p_value < threshold:
            return mark
    return ""


REGRESSORS = {
    "DIVP": "divp",
    "DIVM": "divm",
    "INF": "inflation_rate_pct",
    "EXR": "exchange_rate",
    "log(FDI)": "log_fdi",
    "SHOCK": "shock",
}

frame_2024 = pd.read_csv(FRAME_2024_IN).set_index("year")
frame_2023 = pd.read_csv(FRAME_2023_IN).set_index("year")
assert frame_2024.shape[0] == 35, f"expected 35-row 1990-2024 frame, got {frame_2024.shape[0]}"
assert frame_2023.shape[0] == 34, f"expected 34-row 1990-2023 frame, got {frame_2023.shape[0]}"


def fit_first_diff_ols(frame: pd.DataFrame, regressors: dict) -> tuple:
    # First-differenced OLS: DeltaERI = b0 + sum(b_k * DeltaX_k) + e, plus Newey-West HAC.
    eri = frame["eri"].rename("ERI")
    X = frame[list(regressors.values())].rename(columns={v: k for k, v in regressors.items()})
    diff_frame = pd.concat([eri, X], axis=1).diff().dropna()
    y = diff_frame["ERI"]
    Xd = diff_frame.drop(columns="ERI")
    design = add_constant(Xd, has_constant="add")
    model = sm.OLS(y, design).fit()
    maxlags = int(np.floor(4 * (model.nobs / 100) ** (2 / 9)))
    hac = model.get_robustcov_results(cov_type="HAC", maxlags=maxlags)
    return model, hac, maxlags


def coef_table(model) -> pd.DataFrame:
    return pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t_stat": model.tvalues.values,
        "p_value": model.pvalues.values,
    }).assign(significance=lambda d: d["p_value"].apply(stars))


def hac_table(model, hac, maxlags) -> pd.DataFrame:
    return pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err_original": model.bse.values,
        "t_stat_original": model.tvalues.values,
        "p_value_original": model.pvalues.values,
        "std_err_hac": hac.bse,
        "t_stat_hac": hac.tvalues,
        "p_value_hac": hac.pvalues,
    }).assign(maxlags=maxlags)


def fit_stats_row(model, package_function="statsmodels.api.OLS") -> pd.DataFrame:
    return pd.DataFrame([{
        "package_function": package_function,
        "n_obs": int(model.nobs),
        "r_squared": model.rsquared,
        "adj_r_squared": model.rsquared_adj,
        "f_statistic": model.fvalue,
        "f_pvalue": model.f_pvalue,
    }])

## Step 0 -- Reload the baseline (primary model) and sanity-check against step vi/vii

Re-fits the first-differenced OLS on the full 1990-2024 sample exactly as step vi did, and checks
the refit against step vi's saved coefficients (and step vii's saved HAC-robust coefficients)
before running any of the checks below -- so a silent spec drift between notebooks fails loudly
rather than quietly changing the robustness comparison.

In [2]:
baseline_model, baseline_hac, baseline_maxlags = fit_first_diff_ols(frame_2024, REGRESSORS)
baseline_coefs = coef_table(baseline_model)
baseline_hac_table = hac_table(baseline_model, baseline_hac, baseline_maxlags)

saved_baseline_coefs = pd.read_csv(BASELINE_COEF_IN).set_index("term")
saved_baseline_hac = pd.read_csv(BASELINE_HAC_IN).set_index("term")

for term in ["DIVP", "DIVM"]:
    refit_coef = baseline_coefs.set_index("term").loc[term, "coef"]
    saved_coef = saved_baseline_coefs.loc[term, "coef"]
    assert np.isclose(refit_coef, saved_coef), f"{term}: refit does not match step vi's saved coefficient"
    refit_hac_p = baseline_hac_table.set_index("term").loc[term, "p_value_hac"]
    saved_hac_p = saved_baseline_hac.loc[term, "p_value_hac"]
    assert np.isclose(refit_hac_p, saved_hac_p), f"{term}: refit HAC p-value does not match step vii's saved value"

print(f"Baseline refit confirmed identical to step vi/vii's saved values. n_obs={int(baseline_model.nobs)}, "
      f"maxlags={baseline_maxlags}.")
print()
print("BASELINE (full 1990-2024 sample, first-differenced OLS, PRIMARY model):")
for term in ["DIVP", "DIVM"]:
    row = baseline_coefs.set_index("term").loc[term]
    hac_row = baseline_hac_table.set_index("term").loc[term]
    print(f"  {term}: coef={row['coef']:.4f}, p_orig={row['p_value']:.4f}{row['significance']}, "
          f"p_hac={hac_row['p_value_hac']:.4f}{stars(hac_row['p_value_hac'])}")

Baseline refit confirmed identical to step vi/vii's saved values. n_obs=34, maxlags=3.

BASELINE (full 1990-2024 sample, first-differenced OLS, PRIMARY model):
  DIVP: coef=2.3529, p_orig=0.0995*, p_hac=0.0573*
  DIVM: coef=-1.4024, p_orig=0.1904, p_hac=0.0915*


## Check 1 -- Restricted to 1990-2023 (34 obs pre-differencing -> 33 post-differencing)

Matches the literal thesis text's stated study period (Ch. 3.2/3.8), dropping the 2024
observation that is itself a flagged deviation (step i). No lag-selection re-run needed -- the
first-differenced OLS has no lag structure to reselect, unlike ARDL.

In [3]:
check1_model, check1_hac, check1_maxlags = fit_first_diff_ols(frame_2023, REGRESSORS)
assert check1_model.nobs == 33, f"expected 33 post-differencing obs, got {int(check1_model.nobs)}"

check1_coefs = coef_table(check1_model)
check1_coefs.to_csv(CHECK1_COEF_OUT, index=False)
fit_stats_row(check1_model).to_csv(CHECK1_FIT_OUT, index=False)
check1_hac_table = hac_table(check1_model, check1_hac, check1_maxlags)
check1_hac_table.to_csv(CHECK1_HAC_OUT, index=False)
print(f"Written -> {CHECK1_COEF_OUT}, {CHECK1_FIT_OUT}, {CHECK1_HAC_OUT}")

print()
print(f"CHECK 1 (1990-2023, n={int(check1_model.nobs)}, maxlags={check1_maxlags}):")
for term in ["DIVP", "DIVM"]:
    row = check1_coefs.set_index("term").loc[term]
    hac_row = check1_hac_table.set_index("term").loc[term]
    print(f"  {term}: coef={row['coef']:.4f}, p_orig={row['p_value']:.4f}{row['significance']}, "
          f"p_hac={hac_row['p_value_hac']:.4f}{stars(hac_row['p_value_hac'])}")

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check1_1990_2023_coefficients.csv, /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check1_1990_2023_fit_stats.csv, /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check1_1990_2023_hac_coefficients.csv

CHECK 1 (1990-2023, n=33, maxlags=3):
  DIVP: coef=2.0858, p_orig=0.1114, p_hac=0.0692*
  DIVM: coef=-0.9737, p_orig=0.3245, p_hac=0.1535


## Check 2 -- Excluding shock-year observations

Drops `SHOCK == 1` observations (2008, 2009, 2020, 2021, 2022), leaving 30 observations (29
post-differencing). **The SHOCK regressor is dropped from this specific re-estimation** rather
than kept as a now-degenerate all-zero column (a constant column is collinear with the intercept
and would make the design matrix singular) -- an explicit, necessary adjustment, not a silent
change of specification. Differencing is applied to the remaining 30 rows in year order after
filtering, matching the plan's stated "30 observations, 29 post-differencing" arithmetic; the two
points that straddle a dropped-year gap (2007->2010, 2019->2023) are differenced across that gap
like any other adjacent pair in the filtered series, since the plan does not call for a
gap-aware treatment.

In [4]:
frame_noshock = frame_2024[frame_2024["shock"] == 0]
assert frame_noshock.shape[0] == 30, f"expected 30 non-shock rows, got {frame_noshock.shape[0]}"
print(f"Non-shock years retained ({len(frame_noshock)}): {frame_noshock.index.tolist()}")
print("SHOCK regressor DROPPED from this re-estimation (degenerate all-zero column after filtering).")

REGRESSORS_NO_SHOCK = {k: v for k, v in REGRESSORS.items() if k != "SHOCK"}
check2_model, check2_hac, check2_maxlags = fit_first_diff_ols(frame_noshock, REGRESSORS_NO_SHOCK)
assert check2_model.nobs == 29, f"expected 29 post-differencing obs, got {int(check2_model.nobs)}"

check2_coefs = coef_table(check2_model)
check2_coefs.to_csv(CHECK2_COEF_OUT, index=False)
fit_stats_row(check2_model).to_csv(CHECK2_FIT_OUT, index=False)
check2_hac_table = hac_table(check2_model, check2_hac, check2_maxlags)
check2_hac_table.to_csv(CHECK2_HAC_OUT, index=False)
print(f"Written -> {CHECK2_COEF_OUT}, {CHECK2_FIT_OUT}, {CHECK2_HAC_OUT}")

print()
print(f"CHECK 2 (excluding shock years, n={int(check2_model.nobs)}, maxlags={check2_maxlags}, SHOCK dropped):")
for term in ["DIVP", "DIVM"]:
    row = check2_coefs.set_index("term").loc[term]
    hac_row = check2_hac_table.set_index("term").loc[term]
    print(f"  {term}: coef={row['coef']:.4f}, p_orig={row['p_value']:.4f}{row['significance']}, "
          f"p_hac={hac_row['p_value_hac']:.4f}{stars(hac_row['p_value_hac'])}")

Non-shock years retained (30): [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2023, 2024]
SHOCK regressor DROPPED from this re-estimation (degenerate all-zero column after filtering).
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check2_excl_shock_coefficients.csv, /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check2_excl_shock_fit_stats.csv, /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check2_excl_shock_hac_coefficients.csv

CHECK 2 (excluding shock years, n=29, maxlags=3, SHOCK dropped):
  DIVP: coef=2.1092, p_orig=0.1220, p_hac=0.0717*
  DIVM: coef=-0.1294, p_orig=0.8984, p_hac=0.7950


## Check 3 -- Using raw FDI instead of log(FDI)

The direct counter-check on the step-i log-transform deviation: re-estimates the first-differenced
OLS with `Δfdi_net_inflows_usd` in place of `Δlog_fdi`, full 1990-2024 sample, all else unchanged.

In [5]:
REGRESSORS_RAW_FDI = {
    "DIVP": "divp", "DIVM": "divm", "INF": "inflation_rate_pct",
    "EXR": "exchange_rate", "FDI": "fdi_net_inflows_usd", "SHOCK": "shock",
}
check3_model, check3_hac, check3_maxlags = fit_first_diff_ols(frame_2024, REGRESSORS_RAW_FDI)
assert check3_model.nobs == 34, f"expected 34 post-differencing obs, got {int(check3_model.nobs)}"

check3_coefs = coef_table(check3_model)
check3_coefs.to_csv(CHECK3_COEF_OUT, index=False)
fit_stats_row(check3_model).to_csv(CHECK3_FIT_OUT, index=False)
check3_hac_table = hac_table(check3_model, check3_hac, check3_maxlags)
check3_hac_table.to_csv(CHECK3_HAC_OUT, index=False)
print(f"Written -> {CHECK3_COEF_OUT}, {CHECK3_FIT_OUT}, {CHECK3_HAC_OUT}")

print()
print(f"CHECK 3 (raw FDI, n={int(check3_model.nobs)}, maxlags={check3_maxlags}):")
for term in ["DIVP", "DIVM"]:
    row = check3_coefs.set_index("term").loc[term]
    hac_row = check3_hac_table.set_index("term").loc[term]
    print(f"  {term}: coef={row['coef']:.4f}, p_orig={row['p_value']:.4f}{row['significance']}, "
          f"p_hac={hac_row['p_value_hac']:.4f}{stars(hac_row['p_value_hac'])}")

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check3_raw_fdi_coefficients.csv, /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check3_raw_fdi_fit_stats.csv, /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_check3_raw_fdi_hac_coefficients.csv

CHECK 3 (raw FDI, n=34, maxlags=3):
  DIVP: coef=1.5176, p_orig=0.2782, p_hac=0.1451
  DIVM: coef=-0.9637, p_orig=0.3672, p_hac=0.1402


## Check 4 -- Alternate ARDL lag length: N/A for the primary-model table

The first-differenced OLS (the primary model) has no lag structure to vary, so this check does
not apply to it -- **explicitly marked N/A below, not silently omitted**, per the plan.

ARDL(1,1)'s lag-length sensitivity remains informative as secondary/exploratory context, but it
was already substantially explored in `vi_estimation.ipynb` (grid-search AIC-selected (1,2) vs.
BIC-implied (2,1), `ardl_lag_selection.csv`) and `vii_post_estimation_diagnostics.ipynb` /
`b_model_b_ardl_bounds_testing.ipynb` (leanest capped ARDL(1,1) vs. the grid-search pick, run
specifically as a degrees-of-freedom robustness check) -- both reached the same substantive
conclusion (bounds test inconclusive at every lag combination tried). Re-running it a third time
here is optional per the plan and is **not** done in this notebook, to avoid redundant work that
would not change the already-established finding; the existing results remain available in
`ardl_lag_selection.csv`, `ardl_bounds_test.csv`, and `ardl_capped_1_1_bounds_test.csv`.

The optional stacked "kitchen sink" combined check (checks 1-3 applied simultaneously) is also
**not run** here -- the plan marks it optional, and running checks one at a time (as done above)
is what isolates which specific condition drives any coefficient movement.

## Comparison table -- baseline vs. checks 1-3 vs. check 4 (N/A)

DIVP and DIVM coefficients, original and HAC-robust p-values/significance, and N, side by side.

In [6]:
def row_from(label, model, hac_tbl, note):
    coefs = coef_table(model).set_index("term")
    hac_idx = hac_tbl.set_index("term")
    return {
        "check": label,
        "n_obs": int(model.nobs),
        "divp_coef": coefs.loc["DIVP", "coef"],
        "divp_p_original": coefs.loc["DIVP", "p_value"],
        "divp_sig_original": coefs.loc["DIVP", "significance"],
        "divp_p_hac": hac_idx.loc["DIVP", "p_value_hac"],
        "divp_sig_hac": stars(hac_idx.loc["DIVP", "p_value_hac"]),
        "divm_coef": coefs.loc["DIVM", "coef"],
        "divm_p_original": coefs.loc["DIVM", "p_value"],
        "divm_sig_original": coefs.loc["DIVM", "significance"],
        "divm_p_hac": hac_idx.loc["DIVM", "p_value_hac"],
        "divm_sig_hac": stars(hac_idx.loc["DIVM", "p_value_hac"]),
        "note": note,
    }


comparison_rows = [
    row_from("Baseline (1990-2024, primary model, for reference)", baseline_model, baseline_hac_table,
              "Full sample; primary model per the 2026-07-17 redesignation."),
    row_from("Check 1 -- restricted to 1990-2023", check1_model, check1_hac_table,
              "Drops the 2024 observation (step-i deviation from the literal thesis text)."),
    row_from("Check 2 -- excluding shock-year observations", check2_model, check2_hac_table,
              "SHOCK regressor dropped (degenerate after filtering); 30 obs (29 post-differencing)."),
    row_from("Check 3 -- raw FDI instead of log(FDI)", check3_model, check3_hac_table,
              "Direct counter-check on the step-i log-transform deviation."),
    {
        "check": "Check 4 -- alternate ARDL lag length", "n_obs": np.nan,
        "divp_coef": np.nan, "divp_p_original": np.nan, "divp_sig_original": "N/A",
        "divp_p_hac": np.nan, "divp_sig_hac": "N/A",
        "divm_coef": np.nan, "divm_p_original": np.nan, "divm_sig_original": "N/A",
        "divm_p_hac": np.nan, "divm_sig_hac": "N/A",
        "note": "N/A for the primary-model table -- first-differenced OLS has no lag structure. "
                "ARDL(1,1) lag sensitivity already explored in steps vi/vii (see markdown above); "
                "not re-run here (optional per the plan).",
    },
]

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(COMPARISON_OUT, index=False)
print(f"Written -> {COMPARISON_OUT}")
comparison_table

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/robustness_checks_comparison.csv


,check,n_obs,divp_coef,divp_p_original,divp_sig_original,divp_p_hac,divp_sig_hac,divm_coef,divm_p_original,divm_sig_original,divm_p_hac,divm_sig_hac,note
0,"Baseline (1990-2024, primary model, for refere...",34.0,2.352928,0.099531,*,0.057288,*,-1.402368,0.190354,,0.091547,*,Full sample; primary model per the 2026-07-17 ...
1,Check 1 -- restricted to 1990-2023,33.0,2.085753,0.111392,,0.069243,*,-0.973747,0.324519,,0.153472,,Drops the 2024 observation (step-i deviation f...
2,Check 2 -- excluding shock-year observations,29.0,2.109156,0.121986,,0.071720,*,-0.129388,0.898428,,0.794998,,SHOCK regressor dropped (degenerate after filt...
3,Check 3 -- raw FDI instead of log(FDI),34.0,1.517574,0.278201,,0.145097,,-0.963722,0.367238,,0.140227,,Direct counter-check on the step-i log-transfo...
4,Check 4 -- alternate ARDL lag length,NaN,NaN,NaN,N/A,NaN,N/A,NaN,NaN,N/A,NaN,N/A,N/A for the primary-model table -- first-diffe...


## Written assessment -- does H1/H2's marginal baseline support hold up?

**Short answer: no, not robustly.** The baseline (primary-model) result was already only
marginal -- DIVP significant at the 10% level only (never 5%), DIVM significant only under the
HAC correction and with a sign opposite the hypothesized direction. Across the three checks, that
marginal evidence weakens further or disappears outright; it does not strengthen or stay stable
under any of the three conditions.

**DIVP (H1).**
- Baseline: 10%-level significant under both original SE (p=0.0995) and HAC SE (p=0.0573).
- Check 1 (1990-2023): the **original-SE result crosses out of 10%-level significance**
  (p=0.1114 > 0.10) -- exactly the boundary-crossing the plan flagged as the thing to watch for.
  It survives only under the HAC correction (p=0.0692). The 2024 observation is doing real work
  in keeping DIVP's non-robust-SE result at the 10% threshold.
- Check 2 (excluding shock years): the same pattern as Check 1 -- not significant under original
  SEs (p=0.1220), still 10%-level significant under HAC (p=0.0717).
- Check 3 (raw FDI): DIVP **loses significance entirely, even under HAC** (p=0.1451 original,
  p=0.1451 vs the baseline's 0.0573 -- roughly triples). This is the sharpest movement of the
  three checks: DIVP's marginal 10%-level HAC result in the baseline depends materially on using
  `log(FDI)` rather than raw FDI, not only on the diversification measure itself.

**DIVM (H2).**
- Baseline: not significant under original SEs (p=0.1904); significant at 10% only under HAC
  (p=0.0915), with a **negative** coefficient -- opposite H2's hypothesized positive direction.
- Check 1 (1990-2023): the HAC-only 10% significance is **lost** (p=0.1535).
- Check 2 (excluding shock years): the coefficient **collapses in magnitude**, from -1.4024
  (baseline) to -0.1294 -- roughly a tenth of its baseline size -- and loses all significance
  (p=0.7950 under HAC). This is the single largest movement in the entire robustness battery and
  points to a specific mechanism: DIVM's marginal negative-and-significant (under HAC) reading in
  the baseline is **almost entirely attributable to the shock-year observations**
  (2008-09, 2020-22) rather than a general DIVM-ERI relationship over the full sample.
- Check 3 (raw FDI): also not significant (p=0.1402 under HAC).

**Overall.** Neither H1 nor H2 clears even the 10% bar in a way that is stable across all three
checks. DIVP's marginal support is sensitive to both the study-period extension (Check 1/2) and,
most sharply, to the log-FDI transform itself (Check 3). DIVM's marginal, wrong-signed result is
substantially a shock-year artifact (Check 2) and does not survive the 1990-2023 restriction
either (Check 1). This should be reported in the Chapter 4 draft as an explicit tempering of the
already-weak baseline verdict from `c_shared_requirements_for_both_models.ipynb`'s Requirement 7
table -- the primary model's H1/H2 evidence is not just marginal, it is fragile, and readers
should not come away thinking a 10%-level baseline result is a robust one.

## Decisions & flags (explicit recap)

- Checks 1-3 are run **one at a time**, not stacked, per the plan's default -- isolates which
  specific condition drives any coefficient movement. The optional combined "kitchen sink" check
  is **not run** here.
- **Check 2's SHOCK-regressor drop** is a necessary adjustment (a degenerate all-zero column
  would be collinear with the intercept), flagged explicitly rather than silently handled.
- **Check 4 is explicitly marked N/A** for the primary-model comparison table -- the
  first-differenced OLS has no lag structure. ARDL(1,1)'s own lag sensitivity was already
  substantially explored in steps vi/vii and is not re-run a third time here (optional per the
  plan).
- **Baseline row cites both original and HAC-robust DIVP/DIVM p-values**, per the plan's explicit
  instruction, since step vii already established the HAC check was needed (inconclusive DW) --
  this notebook's comparison table is read against the same standard applied upstream, not a
  weaker one.
- **DIVP crosses the 10%-significance boundary under Check 1's original SEs** (p=0.0995 ->
  0.1114) -- reported as a genuine boundary-crossing per the plan's explicit instruction to watch
  for this, not smoothed into "still roughly 10%."
- **DIVM's coefficient collapses by roughly 90% in magnitude under Check 2** (shock-year
  exclusion) -- flagged as the single largest movement in the battery and attributed to the
  shock-year observations specifically, per the plan's own "is the 2022 crisis year doing a lot
  of work" framing.

## Definition of done

In [7]:
checks_done = {
    "Checks 1-3 run against the first-differenced OLS (primary model)":
        int(check1_model.nobs) == 33 and int(check2_model.nobs) == 29 and int(check3_model.nobs) == 34,
    "Check 4 explicitly marked N/A for the primary-model table, not silently omitted":
        comparison_table.set_index("check").loc["Check 4 -- alternate ARDL lag length", "divp_sig_original"] == "N/A",
    "Single comparison table assembled: baseline (original + HAC p-values) vs. checks 1-3, "
    "DIVP/DIVM coefficients and significance side by side":
        {"divp_coef", "divp_p_original", "divp_p_hac", "divm_coef", "divm_p_original", "divm_p_hac"}.issubset(comparison_table.columns)
        and len(comparison_table) == 5,
    "Written interpretation of coefficient stability across checks, addressing whether DIVP's "
    "marginal 10%-level baseline result holds up": True,
    "Baseline refit verified identical to step vi/vii's saved coefficients and HAC p-values before "
    "any check was run": True,
}

for description, passed in checks_done.items():
    print(("PASS" if passed else "FAIL") + f" -- {description}")

assert all(checks_done.values()), "Definition of done not fully met"

PASS -- Checks 1-3 run against the first-differenced OLS (primary model)
PASS -- Check 4 explicitly marked N/A for the primary-model table, not silently omitted
PASS -- Single comparison table assembled: baseline (original + HAC p-values) vs. checks 1-3, DIVP/DIVM coefficients and significance side by side
PASS -- Written interpretation of coefficient stability across checks, addressing whether DIVP's marginal 10%-level baseline result holds up
PASS -- Baseline refit verified identical to step vi/vii's saved coefficients and HAC p-values before any check was run


## Conclusion

All three applicable robustness checks were run against the first-differenced OLS (the primary
model per the 2026-07-17 redesignation), one at a time, with the baseline row citing both
original and HAC-robust p-values per the plan's instruction. Check 4 (alternate ARDL lag length)
is explicitly marked N/A for the primary-model table -- the first-differenced OLS has no lag
structure -- and is not re-run for ARDL(1,1) either, since steps vi/vii already explored that
question thoroughly and reached the same conclusion (bounds test inconclusive) at every lag
combination tried.

The headline finding: H1 and H2's already-marginal baseline support (10%-level DIVP, HAC-only
wrong-signed DIVM) is **not robust**. DIVP's result depends materially on the log(FDI) transform
(it disappears entirely under raw FDI, Check 3) and, to a lesser extent, on the 2024 extension and
the shock-year observations (its original-SE significance is lost under both Check 1 and Check
2). DIVM's marginal, contrary-signed result is substantially a shock-year artifact -- its
coefficient collapses by roughly 90% once shock years are excluded (Check 2) -- and does not
survive the 1990-2023 restriction either (Check 1). See `robustness_checks_comparison.csv` for the
full numbers and the written assessment above for the per-check narrative, both directly usable
in Chapter 4 draft section 7 and feeding back into section 5's H1/H2 confidence statement.